<a href="https://colab.research.google.com/github/vedaanshi2111-debug/Arraysofobject-/blob/main/minimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Install OpenMM
!pip install openmm
!pip install git+https://github.com/openmm/pdbfixer.git
!pip install nglview


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 95.4 MB/s eta 0:00:00
  Cloning https://github.com/openmm/pdbfixer.git to /tmp/pip-req-build-_stmzg7f
  Running command git clone --filter=blob:none --quiet https://github.com/openmm/pdbfixer.git /tmp/pip-req-build-_stmzg7f
  Resolved https://github.com/openmm/pdbfixer.git to commit 94cfa4c0ca551cdc5f13320f9a658efd59f2b881
  Preparing metadata (setup.py) ... done
  Created wheel for pdbfixer: filename=pdbfixer-1.12.0-py3-none-any.whl size=681683 sha256=a2f039be6456a49488a808e8b3c63ed5e9d8fc8c926780a9a2d71366c0728742
  Stored in directory: /tmp/pip-ephem-wheel-cache-2n5alwd2/wheels/0b/90/9a/de8a860c8290c72a309b03127bbfdf72a3f9a09a7b7faf8882
Successfully built pdbfixer
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.2/26.2 MB 38.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━

In [ ]:
#Upload Protein PDB File
from google.colab import files
uploaded = files.upload()

Saving HEMOGLOBIN.B99990002.pdb to HEMOGLOBIN.B99990002.pdb


In [ ]:
pdb_file = list(uploaded.keys())[0]

In [ ]:
#Fix structure
from pdbfixer import PDBFixer
from openmm.app import PDBFile
fixer = PDBFixer(filename=pdb_file)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(pH=7.0)
PDBFile.writeFile(fixer.topology, fixer.positions, open("fixed.pdb", "w"))
print("✔ Structure fixed")

✔ Structure fixed


In [ ]:
#Energy minimization
from openmm.app import *
from openmm import *
from openmm.unit import *

pdb = PDBFile("fixed.pdb")

forcefield = ForceField('amber14-all.xml', 'amber14/tip3p.xml')

system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=NoCutoff,
    constraints=HBonds
)

integrator = LangevinIntegrator(
    300*kelvin,
    1/picosecond,
    0.002*picoseconds
)

simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)

print("Initial energy:",
      simulation.context.getState(getEnergy=True).getPotentialEnergy())

print("Minimizing energy...")
simulation.minimizeEnergy()

state = simulation.context.getState(getEnergy=True)
print("Final energy:", state.getPotentialEnergy())


Initial energy: 77367.76360994588 kJ/mol
Minimizing energy...
Final energy: -20184.645260013473 kJ/mol


In [ ]:
#Save minimized structure
positions = simulation.context.getState(getPositions=True).getPositions()
with open("minimized.pdb", "w") as f:
    PDBFile.writeFile(simulation.topology, positions, f)

print("✔ Minimized structure saved")
#Download
files.download("minimized.pdb")

✔ Minimized structure saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>